In [76]:
import pandas as pd

### Carregando base de setores básicos

In [77]:
df_ibge = pd.read_csv(
    'Base_de_Dados/Agregados_por_setores_basico_BR.csv',
    sep=';',
    encoding='latin1',
    low_memory=False
)
df_ibge.head(2)

,CD_SETOR,SITUACAO,CD_SIT,CD_TIPO,AREA_KM2,CD_REGIAO,NM_REGIAO,CD_UF,NM_UF,CD_MUN,...,NM_CONCURB,v0001,v0002,v0003,v0004,v0005,v0006,v0007,v0008,v0009
0,110001505000002,Urbana,1,0,"0,5393102",1,Norte,11,Rondônia,1100015,...,NaN,928,376,376,0,"2,8","0,0923",336,11,29
1,110001505000003,Urbana,1,0,"0,2362175",1,Norte,11,Rondônia,1100015,...,NaN,556,243,243,0,"2,7","0,0048",208,8,27


In [78]:
df_setores_slz = (
    df_ibge[df_ibge['NM_MUN'] == 'São Luís']
    [['CD_SETOR', 'AREA_KM2']]
    .copy()
)

df_setores_slz.head(2)

,CD_SETOR,AREA_KM2
52988,211130005000001,"0,0777277"
52989,211130005000002,"0,0877081"


### Carregando bases demograficas

In [79]:
df_d = pd.read_csv(
    'Base_de_Dados/Agregados_por_setores_demografia_BR.csv',
    sep=';',
    encoding='latin1',
    low_memory=False
)

df_demografia = df_d[["CD_setor", "V01006", "V01031"]]
df_demografia["POPULAÇÃO IDOSA"] = pd.to_numeric(df_d["V01040"], errors="coerce") + pd.to_numeric(df_d["V01041"], errors="coerce")

df_demografia.rename(columns={
    'V01006': 'POPULAÇÃO',
    "V01031": "POPULAÇÃO INFANTIL"
}, inplace=True)

df_demografia.head(2)

,CD_setor,POPULAÇÃO,POPULAÇÃO INFANTIL,POPULAÇÃO IDOSA
0,110001505000002,928,68,119.0
1,110001505000003,556,37,81.0


### Filtragem do dataset
Seleção de colunas importantes,
Calculo da densidade habitacional e adição dela como coluna

In [80]:
df_setores_slz['AREA_KM2'] = (
    df_setores_slz['AREA_KM2']
    .str.replace(',', '.', regex=False)
)

df_demografia = df_demografia.rename(
    columns={"CD_setor": "CD_SETOR"}
)

In [81]:
df_setores_slz['AREA_KM2'] = pd.to_numeric(df_setores_slz['AREA_KM2'], errors='coerce')

df_merge = df_setores_slz.merge(
    df_demografia,
    on="CD_SETOR",
)
df_merge.head(2)

,CD_SETOR,AREA_KM2,POPULAÇÃO,POPULAÇÃO INFANTIL,POPULAÇÃO IDOSA
0,211130005000001,0.077728,236,18,53.0
1,211130005000002,0.087708,247,8,58.0


In [82]:
df_merge['DENSIDADE POPULACIONAL (hab/KM²)'] = pd.to_numeric(df_merge['POPULAÇÃO'], errors='coerce') / df_merge['AREA_KM2']
df_merge.head()

,CD_SETOR,AREA_KM2,POPULAÇÃO,POPULAÇÃO INFANTIL,POPULAÇÃO IDOSA,DENSIDADE POPULACIONAL (hab/KM²)
0,211130005000001,0.077728,236,18,53.0,3036.240619
1,211130005000002,0.087708,247,8,58.0,2816.159511
2,211130005000003,0.158956,232,13,40.0,1459.525227
3,211130005000004,0.074461,213,15,35.0,2860.554034
4,211130005000005,0.182872,516,29,116.0,2821.651913


In [83]:
# Para exportar para csv
df_merge.to_csv(
    'Dados_Tratados/densidade_populacional_setores_censitarios.csv',
    sep=';',
    index=False,
    encoding='utf-8-sig'
)